# Stage 05: End-to-End Hybrid Pipeline on Unseen Test Images

## Overview
This notebook executes the full hybrid pipeline on unseen test data:
$$\text{Test Image} \rightarrow \text{U-Net Mask} \rightarrow \text{regionprops} \rightarrow \text{Numeric Summary} \rightarrow \text{LLM JSON Record & Narrative}$$


In [5]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import sys
import gc
from pathlib import Path
import torch
import pandas as pd

CURRENT_DIR = Path.cwd()
PROJECT_ROOT = CURRENT_DIR.parent if CURRENT_DIR.name.lower() == "notebooks" else CURRENT_DIR
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.unet import SmallUNet
from src.pipeline import run_batch_hybrid_pipeline


## 1. Load Pretrained U-Net Model


In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = SmallUNet(features=(16, 32, 64, 128)).to(device)

ckpt_path = PROJECT_ROOT / "outputs" / "models" / "unet_combined_bce___dice.pth"
if ckpt_path.exists():
    state = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(state["model_state_dict"])
    print(f"Loaded trained checkpoint from {ckpt_path.name}")
else:
    print(f"Warning: Checkpoint not found at {ckpt_path}. Using uninitialized model.")


Loaded trained checkpoint from unet_combined_bce___dice.pth


## 2. Execute Batch Hybrid Pipeline


In [7]:
data_dir = PROJECT_ROOT / "data" / "nuclei_dataset"
if not data_dir.exists():
    data_dir = PROJECT_ROOT / "nuclei_dataset"

test_img_dir = data_dir / "test" / "images"
test_mask_dir = data_dir / "test" / "masks"

summary_df, results = run_batch_hybrid_pipeline(
    model=model,
    image_dir=test_img_dir,
    mask_dir=test_mask_dir,
    output_csv_path=PROJECT_ROOT / "outputs" / "csv" / "hybrid_pipeline_test_summary.csv",
    llm_model="llama3.2",
    device=device
)
display(summary_df[["image_id", "dataset_density_regime", "predicted_density_class", "ground_truth_nuclei_count", "predicted_component_count", "count_error", "absolute_count_error", "llm_n_objects", "llm_measurement_audit_match", "json_valid", "quality_flag", "dice_vs_gt", "iou_vs_gt"]])


Running Hybrid Pipeline across 12 images in images...
  [01/12] test_000.png: Objects=8, Density=sparse, Quality=pass
  [02/12] test_001.png: Objects=14, Density=normal, Quality=pass
  [03/12] test_002.png: Objects=25, Density=normal, Quality=pass
  [04/12] test_003.png: Objects=19, Density=normal, Quality=pass
  [05/12] test_004.png: Objects=42, Density=clustered, Quality=pass
  [06/12] test_005.png: Objects=15, Density=normal, Quality=pass
  [07/12] test_006.png: Objects=8, Density=sparse, Quality=pass
  [08/12] test_007.png: Objects=23, Density=normal, Quality=pass
  [09/12] test_008.png: Objects=23, Density=normal, Quality=pass
  [10/12] test_009.png: Objects=21, Density=normal, Quality=pass
  [11/12] test_010.png: Objects=34, Density=normal, Quality=pass
  [12/12] test_011.png: Objects=18, Density=normal, Quality=pass
Saved aggregated test summary to C:\Users\LOJANA-UOH\OneDrive - University of Hertfordshire\DAwAI\Assignment-03\AI imaging Coding\outputs\csv\hybrid_pipeline_test_su

,image_id,dataset_density_regime,predicted_density_class,ground_truth_nuclei_count,predicted_component_count,count_error,absolute_count_error,llm_n_objects,llm_measurement_audit_match,json_valid,quality_flag,dice_vs_gt,iou_vs_gt
0,test_000,sparse,sparse,8,8,0,0,8,True,True,pass,0.9935,0.9871
1,test_001,normal,normal,19,14,-5,5,14,True,True,pass,0.9850,0.9705
2,test_002,normal,normal,31,25,-6,6,25,True,True,pass,0.9913,0.9827
3,test_003,normal,normal,21,19,-2,2,19,True,True,pass,0.9869,0.9742
4,test_004,dense,clustered,75,42,-33,33,42,True,True,pass,0.9926,0.9854
5,test_005,clustered,normal,49,15,-34,34,15,True,True,pass,0.9928,0.9858
6,test_006,sparse,sparse,8,8,0,0,8,True,True,pass,0.9820,0.9646
7,test_007,normal,normal,30,23,-7,7,23,True,True,pass,0.9920,0.9842
8,test_008,normal,normal,28,23,-5,5,23,True,True,pass,0.9932,0.9864
9,test_009,normal,normal,24,21,-3,3,21,True,True,pass,0.9894,0.9791


## 3. Summary & Audit Verification


In [8]:
mean_dice = summary_df["dice_vs_gt"].mean()
std_dice = summary_df["dice_vs_gt"].std()

mean_iou = summary_df["iou_vs_gt"].mean()
std_iou = summary_df["iou_vs_gt"].std()

mae_count = summary_df["absolute_count_error"].mean() if "absolute_count_error" in summary_df.columns else None
audit_col = "llm_measurement_audit_match" if "llm_measurement_audit_match" in summary_df.columns else "audit_count_match"

print(f"Total test images processed: {len(summary_df)}")
print(f"Mean test Dice: {mean_dice:.4f} ± {std_dice:.4f}")
print(f"Mean test IoU: {mean_iou:.4f} ± {std_iou:.4f}")
if mae_count is not None:
    print(f"Mean absolute count error (MAE): {mae_count:.2f} nuclei")
print(f"LLM measurement reproduction match: {(summary_df[audit_col].sum() / len(summary_df))*100:.1f}%")
print(f"Schema validity rate (json_valid): {(summary_df['json_valid'].sum() / len(summary_df))*100:.1f}%")


Total test images processed: 12
Mean test Dice: 0.9891 ± 0.0041
Mean test IoU: 0.9784 ± 0.0080
Mean absolute count error (MAE): 12.58 nuclei
LLM measurement reproduction match: 100.0%
Schema validity rate (json_valid): 100.0%
